# Understanding Supervised Learning Problem Types

## Before you train your first model, you must understand what kind of problem you are solving.

This is **not a minor detail**. It is the **foundation** that determines:
- Which algorithms are appropriate
- Which metrics are meaningful  
- How success itself is defined

> **Key Insight**: Supervised learning is not a single technique. It is a *family of problem types*, each with distinct characteristics, evaluation strategies, and real-world applications.

By the end of this notebook, when someone describes a prediction task to you, you will immediately know:
- ✓ Is this classification or regression?
- ✓ Binary or multi-class?
- ✓ Balanced or imbalanced?
- ✓ What does success look like?

These are not abstract questions. **They are the first questions you must answer before writing any code.**

## What is Supervised Learning?

Supervised learning is a machine learning paradigm where you train a model on **labeled data** — data where each example has both input features **and** a known, correct output.

The model learns the relationship between inputs and outputs by observing thousands of examples, then uses that learned relationship to predict outputs for new, unseen inputs.

### The Fundamental Formula

```
Input Features (X) + Correct Answers (y) → Training → Model → Predictions on New Data
```

The word **"supervised"** comes from the idea that the **correct answers supervise the learning process**.

The model:
1. Makes a prediction
2. Compares it to the true answer
3. Measures how wrong it was
4. Adjusts itself to be less wrong next time

This cycle repeats across all training examples, iterations, until the model converges.

### Supervised vs Other Learning Paradigms

| Paradigm | Characteristics | Example |
|----------|-----------------|---------|
| **Supervised** | You have labeled data (correct answers known) | Predicting stock price from historical data |
| **Unsupervised** | You have only input data, no labels | Clustering customers by similarity |
| **Reinforcement** | Agent learns by trial/error, receives rewards/penalties | AlphaGo learning to play Go |

**In this course**: You work exclusively with supervised learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_regression, make_classification

# Example: Simple labeled dataset for supervised learning
# Input Features (X) + Correct Answers (y)

print("=" * 60)
print("SUPERVISED LEARNING: LABELED DATA EXAMPLE")
print("=" * 60)

# Create a simple classification dataset
np.random.seed(42)
n_samples = 100

# Features: Student study hours and practice problems completed  
study_hours = np.random.uniform(1, 10, n_samples)
practice_problems = np.random.uniform(0, 50, n_samples)

# Target: Pass/Fail (1 = Pass, 0 = Fail)
# Rule: If (study_hours > 4) AND (practice_problems > 20), then Pass
exam_passed = ((study_hours > 4) & (practice_problems > 20)).astype(int)

# Create DataFrame
df_labeled = pd.DataFrame({
    'study_hours': study_hours,
    'practice_problems': practice_problems,
    'exam_passed': exam_passed  # The label (correct answer)
})

print("\nLabeled Dataset (first 10 rows):")
print(df_labeled.head(10))
print(f"\nTarget variable (y):\n{df_labeled['exam_passed'].value_counts()}")
print(f"- Passed: {(df_labeled['exam_passed']==1).sum()} students")
print(f"- Failed: {(df_labeled['exam_passed']==0).sum()} students")

---

# The Two Core Problem Types: Classification vs Regression

## The Critical Distinction

Supervised learning divides into **two fundamental problem types** based on the **nature of the target variable** you are predicting.

| Aspect | **Classification** | **Regression** |
|--------|-------------------|----------------|
| **Output Type** | Discrete category (label) | Continuous number |
| **Examples** | Spam/Not Spam, Disease/No Disease | House price, Temperature, Stock return |
| **Output Space** | Finite set {A, B, C, ...} | Unbounded continuous range |
| **Loss Functions** | Cross-entropy, log loss, hinge | MSE, MAE |
| **Metrics** | Accuracy, Precision, Recall, F1, ROC-AUC | RMSE, MAE, R², MAPE |
| **Model Output** | Class label + probabilities | Numerical prediction ± interval |
| **What "Good" Means** | High accuracy on right class | Low prediction error |

### The First Question You Must Ask

**Am I predicting a category or a continuous value?**

Getting this wrong is a **fundamental error** that invalidates everything that follows.

## Classification: Predicting Categories

Classification is the task of predicting which **category or class** an example belongs to. The output is **discrete** — a label from a **fixed set of possibilities**.

### Examples of Classification

- **Is this email spam?** Binary (2 classes): Spam vs Not Spam
- **Which species is this flower?** Multi-class (3 classes): Iris Setosa, Versicolor, Virginica
- **Which digit is in this image?** Multi-class (10 classes): 0, 1, 2, ..., 9
- **Which diseases does this patient have?** Multi-label: Multiple diseases simultaneously

### Key Characteristic

**The output is a category. There is no notion of "how much".**

- An email IS spam or ISN'T spam (not "half spam")
- A flower IS setosa, versicolor, or virginica (no ordering)
- The model learns **decision boundaries** that separate categories in feature space

### Classification Model Output

A classification model produces:

1. **A predicted class label** (the category with highest probability)
2. **A probability distribution** over all classes (confidence in each option)

Example output for spam classification:
```
Prediction: Spam (class 1)
Probabilities: {Not Spam: 0.15, Spam: 0.85}
```

**The 85% confidence is fundamentally different from 51% confidence**, even though both result in "Spam" prediction.

In [ ]:
print("=" * 70)
print("CLASSIFICATION EXAMPLE: PREDICTING STOCK PRICE DIRECTION")
print("=" * 70)

# Create synthetic stock data
np.random.seed(42)
n_stocks = 50

# Features
price_change_pct = np.random.normal(0, 5, n_stocks)  # % change
volume_change = np.random.normal(1, 0.3, n_stocks)   # volume multiplier
rsi = np.random.uniform(20, 80, n_stocks)             # momentum indicator

# Classification target: Will price go UP or DOWN next day?
# Rule: If (price_change > 0) AND (RSI < 70), then UP
stock_direction = ((price_change_pct > 0) & (rsi < 70)).astype(int)
# 1 = UP, 0 = DOWN

df_stocks = pd.DataFrame({
    'price_change': price_change_pct,
    'volume_change': volume_change,
    'rsi': rsi,
    'next_day_direction': stock_direction  # Classification target
})

print("\nStock Dataset (first 5 rows):")
print(df_stocks.head())

print("\n" + "=" * 70)
print("CLASSIFICATION TARGET VARIABLE ANALYSIS")
print("=" * 70)
print(f"\nTarget variable: 'next_day_direction'")
print(f"Data type: {df_stocks['next_day_direction'].dtype}")
print(f"Unique values: {sorted(df_stocks['next_day_direction'].unique())}")
print(f"Distribution:")
print(f"  - DOWN (0): {(df_stocks['next_day_direction']==0).sum()} days ({(df_stocks['next_day_direction']==0).sum()/len(df_stocks)*100:.1f}%)")
print(f"  - UP (1): {(df_stocks['next_day_direction']==1).sum()} days ({(df_stocks['next_day_direction']==1).sum()/len(df_stocks)*100:.1f}%)")

print("\n✓ CLASSIFICATION CHARACTERISTICS:")
print("  - Discrete output: Only 2 possible values (0 or 1)")
print("  - No continuous magnitude: Can't be 0.5 (neither 100% down nor up)")
print("  - Probability-based: Model outputs probability for each class")

---

## Regression: Predicting Continuous Values

Regression is the task of predicting a **continuous numerical value**. The output is **not a category** but **a number on a continuous scale**.

### Examples of Regression

- **What will this house sell for?** Predicting price: $310,500, $285,000, etc.
- **How many units will we sell next month?** Predicting count: 1,245 units
- **What will the temperature be tomorrow?** Predicting degrees: 23.5°C
- **How long will this delivery take?** Predicting time: 37.5 minutes

### Key Characteristic

**The output is a number. There is a notion of magnitude and distance.**

- $310,000 is much closer to $285,000 than to $500,000
- The model learns a **function** that maps input features to continuous output
- Small changes in inputs → small changes in output (usually)

### Regression Model Output

A regression model produces:

1. **A single numerical prediction** for each example
2. **Optionally, a confidence interval** indicating the range of plausible values

Example output for house price prediction:
```
Prediction: $285,000
95% Confidence Interval: [$270,000 - $300,000]
```

**This tells you the point estimate AND the uncertainty around it.**

In [ ]:
print("=" * 70)
print("REGRESSION EXAMPLE: PREDICTING HOUSE PRICES")
print("=" * 70)

# Create synthetic house data
np.random.seed(42)
n_houses = 100

# Features
square_footage = np.random.uniform(1000, 5000, n_houses)
age_years = np.random.uniform(0, 50, n_houses)
bedrooms = np.random.randint(1, 6, n_houses)

# Regression target: Selling price (continuous number)
# Price = base + (sqft * price_per_sqft) + (age * age_effect) + (bedrooms * bed_effect)
price = 100000 + (square_footage * 150) + (age_years * -1000) + (bedrooms * 25000)
price += np.random.normal(0, 50000, n_houses)  # Add noise

df_houses = pd.DataFrame({
    'square_footage': square_footage,
    'age_years': age_years,
    'bedrooms': bedrooms,
    'selling_price': price  # Regression target (continuous)
})

print("\nHouse Dataset (first 5 rows):")
print(df_houses.head())

print("\n" + "=" * 70)
print("REGRESSION TARGET VARIABLE ANALYSIS")
print("=" * 70)
print(f"\nTarget variable: 'selling_price'")
print(f"Data type: {df_houses['selling_price'].dtype}")
print(f"Min price: ${df_houses['selling_price'].min():,.0f}")
print(f"Max price: ${df_houses['selling_price'].max():,.0f}")
print(f"Mean price: ${df_houses['selling_price'].mean():,.0f}")
print(f"Number of unique values: {df_houses['selling_price'].nunique()}")
print(f"  (Nearly every price is unique - this is regression!)")

print("\n✓ REGRESSION CHARACTERISTICS:")
print("  - Continuous output: Infinite possible values (not limited to categories)")
print("  - Has magnitude: $285k is different from $500k")
print("  - Distance matters: $280k is closer to $290k than to $500k")
print("  - Float values: Prices include decimals (e.g., $285,237.50)")

# Visualize the difference
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Classification - discrete categories
ax1.scatter(df_stocks['price_change'], df_stocks['rsi'], 
           c=df_stocks['next_day_direction'], cmap='RdYlGn', s=100, alpha=0.6, edgecolors='black')
ax1.set_xlabel('Price Change (%)', fontsize=11, fontweight='bold')
ax1.set_ylabel('RSI (Momentum)', fontsize=11, fontweight='bold')
ax1.set_title('CLASSIFICATION: Discrete Categories\n(UP=1 or DOWN=0)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
cbar1 = plt.colorbar(ax1.collections[0], ax=ax1)
cbar1.set_label('Direction', fontsize=10)

# Regression - continuous values
scatter = ax2.scatter(df_houses['square_footage'], df_houses['selling_price'], 
                      c=df_houses['selling_price'], cmap='viridis', s=100, alpha=0.6, edgecolors='black')
ax2.set_xlabel('Square Footage', fontsize=11, fontweight='bold')
ax2.set_ylabel('Selling Price ($)', fontsize=11, fontweight='bold')
ax2.set_title('REGRESSION: Continuous Values\n(Almost infinite possible prices)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
cbar2 = plt.colorbar(scatter, ax=ax2)
cbar2.set_label('Price ($)', fontsize=10)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

print("\nVisualization shows the fundamental difference:")
print("  LEFT: Classification = Discrete boundaries between classes")
print("  RIGHT: Regression = Continuous relationship (smooth gradient)")

---

# Classification Subtypes

## Binary Classification (2 Classes)

Binary classification is the **simplest and most common** form. The target variable has **exactly two classes**.

### Binary Classification Examples

- **Spam detection**: Spam vs Not Spam
- **Fraud detection**: Fraudulent vs Legitimate
- **Medical diagnosis**: Disease Present vs Disease Absent
- **Customer churn**: Will Churn vs Will Not Churn
- **Loan approval**: Approve vs Reject

### Positive vs Negative Class Framing

Binary problems are often framed as **positive class vs negative class**:

| | **Positive Class (1)** | **Negative Class (0)** |
|---|---|---|
| **Meaning** | The thing you're looking for | The absence of that thing |
| **Examples** | Spam, Fraud, Disease, Churn | Not Spam, Legitimate, Healthy, Retained |
| **Why it matters** | Metrics (precision, recall, F1) are defined relative to positive class | False positives vs false negatives have different costs |

### Key Characteristic

The model outputs a **single probability** — the probability of the positive class.

If that probability > threshold (usually 0.5) → Predict positive class  
Otherwise → Predict negative class

In [ ]:
print("=" * 70)
print("BINARY CLASSIFICATION EXAMPLE: CREDIT DEFAULT PREDICTION")
print("=" * 70)

# Create synthetic credit decision data
np.random.seed(42)
n_customers = 200

# Features
credit_score = np.random.normal(650, 100, n_customers)  # 300-850 scale
income_k = np.random.exponential(50, n_customers) + 20  # $20k-infinity
debt_ratio = np.random.uniform(0.1, 1.0, n_customers)   # 10%-100%

# Binary target: Will customer DEFAULT on loan? (0=No default, 1=Default)
# Rule: If (credit_score < 600) OR (debt_ratio > 0.7), then likely to default
default_likelihood = (credit_score < 600).astype(float) * 0.4 + (debt_ratio > 0.7).astype(float) * 0.4
loan_default = (np.random.random(n_customers) < default_likelihood).astype(int)

df_credit = pd.DataFrame({
    'credit_score': credit_score,
    'income_k': income_k,
    'debt_ratio': debt_ratio,
    'defaulted': loan_default  # Binary target: 0 or 1
})

print("\nCredit Dataset (first 8 rows):")
print(df_credit.head(8))

print("\n" + "=" * 70)
print("BINARY CLASSIFICATION ANALYSIS")
print("=" * 70)
print(f"\nTarget variable: 'defaulted' (loan default?)")
print(f"Classes: {sorted(df_credit['defaulted'].unique())}")
print(f"\nClass distribution:")
negatives = (df_credit['defaulted']==0).sum()
positives = (df_credit['defaulted']==1).sum()
print(f"  - Negative Class (0, No Default): {negatives} customers ({negatives/len(df_credit)*100:.1f}%)")
print(f"  - Positive Class (1, Default):    {positives} customers ({positives/len(df_credit)*100:.1f}%)")

print("\n✓ BINARY CLASSIFICATION CHARACTERISTICS:")
print("  - Exactly 2 classes: 0 (negative) and 1 (positive)")
print("  - Model outputs single probability: P(default = 1)")
print("  - Decision threshold: If P(default) > 0.5 → Predict 'will default'")
print(f"  - Class imbalance: {max(negatives, positives)/min(negatives, positives):.1f}x imbalance")
print("    ⚠️  In real world, defaults are rare (imbalanced!)")

---

## Multi-Class Classification (3+ Mutually Exclusive Classes)

Multi-class classification extends binary classification to **more than two classes**, where each class is **mutually exclusive**.

### Examples

- **Iris species classification**: Setosa, Versicolor, Virginica (3 classes)
- **Handwritten digit recognition**: 0, 1, 2, ..., 9 (10 classes)
- **Document categorization**: Sports, Politics, Technology, Entertainment
- **Customer segment prediction**: Budget, Mid-tier, Premium
- **Market regime**: Bear Market, Sideways, Bull Market

### Key Constraint

**Mutually exclusive classes**: Each example belongs to **exactly one** class.

- A flower cannot be both Setosa AND Versicolor
- A digit cannot be both 3 AND 7
- A market cannot simultaneously be Bear AND Bull

### Model Output

Model outputs **probability distribution over all classes** (sums to 1.0):

```
Example: Iris species prediction
True class: Versicolor
Predicted probabilities: {Setosa: 0.05, Versicolor: 0.80, Virginica: 0.15}
Predicted class: Versicolor (highest probability)
```

In [ ]:
print("=" * 70)
print("MULTI-CLASS EXAMPLE: MARKET REGIME CLASSIFICATION (3 CLASSES)")
print("=" * 70)

# Load Iris dataset as example of multi-class classification
from sklearn.datasets import load_iris

iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = iris.target_names[iris.target]

print("\nIris Dataset (Multi-class Classification - Flower Species):")
print(df_iris.head(10))

print("\n" + "=" * 70)
print("MULTI-CLASS ANALYSIS")
print("=" * 70)
print(f"\nTarget variable: 'species' (flower type)")
print(f"Number of classes: {len(df_iris['species'].unique())}")
print(f"Classes: {list(df_iris['species'].unique())}")

print(f"\nClass distribution:")
for cls in df_iris['species'].unique():
    count = (df_iris['species'] == cls).sum()
    pct = count / len(df_iris) * 100
    print(f"  - {cls}: {count} samples ({pct:.1f}%)")

print("\n✓ MULTI-CLASS CHARACTERISTICS:")
print("  - 3+ mutually exclusive classes: {Setosa, Versicolor, Virginica}")
print("  - Model outputs probability for EACH class (sums to 1.0)")
print("  - Example: P(Setosa)=0.1, P(Versicolor)=0.8, P(Virginica)=0.1 → sum=1.0")
print("  - Metrics: Use macro/micro/weighted averaging across classes")

# Show probability distribution example
print("\nExample model prediction probabilities:")
print("Sample 1: P(Setosa)=0.05, P(Versicolor)=0.85, P(Virginica)=0.10 → Predict: Versicolor")
print("Sample 2: P(Setosa)=0.15, P(Versicolor)=0.40, P(Virginica)=0.45 → Predict: Virginica")

---

# How to Identify Problem Type from Business Requirements

## The Four Critical Questions

When given a real-world ML task (often in business language), you must translate it to the technical problem type.

### Question 1: What am I predicting?

**Is the output...**
- A **category** from a fixed set? → **Classification**
- A **number** on a continuous scale? → **Regression**

| Business Statement | Type | Reasoning |
|---|---|---|
| "Predict which customers will cancel" | Classification | Churn: Yes/No (binary) |
| "Predict how much revenue each customer generates" | Regression | Dollar amount (continuous) |
| "Predict which product category user will buy" | Classification | Category from fixed set |
| "Predict if patient has diabetes" | Classification | Yes/No (binary) |

### Question 2: How many possible outcomes?

**For Classification:**
- 2 classes → Binary classification
- 3+ mutually exclusive → Multi-class
- Multiple can apply → Multi-label

**For Regression:**
- Unbounded continuous → Standard regression
- Non-negative integers → Count regression (0, 1, 2, ...)
- Bounded range (0-100%) → Bounded regression

### Question 3: What does target variable look like in data?

**Examine the actual data:**

```python
# Classification target
df['churn'].unique()
→ array(['Yes', 'No'])

# Regression target
df['price'].head()
→ [180000, 235000, 195000, 275000, 189500]
```

**The data type and distribution tell you immediately:**

| Target | Type | Characteristics |
|---|---|---|
| {'Yes', 'No'} | Classification | Strings/categories, small # unique values |
| {0, 1, 2, 3} | Classification | Integers, represent categories (not magnitude) |
| [180000.5, 195230.2, ...] | Regression | Floats, large # unique values, continuous range |

### Question 4: What does success look like?

**How will the model be evaluated?**

| Success Criterion | Type | Metric |
|---|---|---|
| "Identify as many fraud cases as possible" | Classification | Optimize Recall |
| "Minimize false alarms" | Classification | Optimize Precision |
| "Predictions within $10k of real price" | Regression | Optimize MAE/RMSE |
| "Rank customers by risk score" | Classification | Optimize ROC-AUC |

**The business success criterion determines the technical metric you optimize for.**

In [ ]:
print("=" * 80)
print("PRACTICAL EXERCISE: IDENTIFYING PROBLEM TYPES FROM BUSINESS REQUIREMENTS")
print("=" * 80)

# SCENARIO A: Bank Credit Risk Prediction
print("\n" + "=" * 80)
print("SCENARIO A: Bank requests credit risk prediction system")
print("=" * 80)

print("\nBusiness Requirement:")
print("  'We want to predict the credit risk score (300-850) for loan applicants'")

print("\n" + "-" * 80)
print("SOLVING WITH 4 QUESTIONS:")
print("-" * 80)

print("\n1️⃣  What am I predicting?")
print("   → A number on a continuous scale (credit score 300-850)")
print("   → **REGRESSION**")

print("\n2️⃣  How many possible outcomes?")
print("   → Continuous range 300-850")
print("   → **Standard regression (unbounded continuous)**")

print("\n3️⃣  What does target look like?")
print("   → Data type: Float/Integer")
print("   → Unique values: Hundreds of different scores")
score_dist = [350, 620, 580, 740, 650, 510, 695, 480, 615]
print(f"   →Example values: {score_dist}")

print("\n4️⃣  What does success mean?")
print("   → 'Predictions within 50 points of actual score'")
print("   → Metric: **MAE (Mean Absolute Error) or RMSE**")

print("\n✓ FINAL ANSWER:")
print("   Problem Type: **REGRESSION**")
print("   Target Variable: Credit score (continuous 300-850)")
print("   Success Metric: MAE ≤ 50 points")
print("   Algorithms: Linear Regression, Ridge, Random Forest Regressor, XGBoost Regressor")

# SCENARIO B: Hospital Readmission Prevention
print("\n\n" + "=" * 80)
print("SCENARIO B: Hospital wants to prevent patient readmissions")
print("=" * 80)

print("\nBusiness Requirement:")
print("  'Predict which patients will be readmitted within 30 days of discharge'")

print("\n" + "-" * 80)
print("SOLVING WITH 4 QUESTIONS:")
print("-" * 80)

print("\n1️⃣  What am I predicting?")
print("   → Which patients will be readmitted (Yes/No)")
print("   → **CLASSIFICATION**")

print("\n2️⃣  How many possible outcomes?")
print("   → Exactly 2: Readmitted or Not Readmitted")
print("   → **Binary classification**")

print("\n3️⃣  What does target look like?")
print("   → Data type: Yes/No or 0/1")
print("   → Unique values: Only 2")
data_ex = pd.DataFrame({
    'readmitted': ['No', 'Yes', 'No', 'No', 'Yes', 'No', 'No'],
    'age': [65, 48, 72, 55, 62, 40, 58]
})
print("   → Example:")
print(data_ex.head(3).to_string(index=False))

print("\n4️⃣  What does success mean?")
print("   → 'Identify as many readmissions as possible'")
print("   → Metric: **HIGH RECALL (catch true readmissions)**")

print("\n✓ FINAL ANSWER:")
print("   Problem Type: **BINARY CLASSIFICATION**")
print("   Target Variable: Readmitted (Yes/No)")
print("   Positive Class: Readmitted")
print("   Success Metric: Recall ≥ 85% (catch most readmissions)")
print("   Algorithms: Logistic Regression, Random Forest, Gradient Boosting")
print("   Evaluation: Track Sensitivity (Recall), Precision, ROC-AUC")

In [ ]:
# SCENARIO C: E-commerce Purchase Forecasting
print("\n\n" + "=" * 80)
print("SCENARIO C: E-commerce wants purchase volume forecasting")
print("=" * 80)

print("\nBusiness Requirement:")
print("  'How many items will a customer purchase in the next month?'")

print("\n1️⃣  What am I predicting?")
print("   → Number of items (can't be negative, integer values)")
print("   → **REGRESSION** (specifically: COUNT regression)")

print("\n2️⃣  How many outcomes?")
print("   → Non-negative integers: 0, 1, 2, 3, ..., infinity")
print("   → Usually many zeros")

print("\n3️⃣  Target variable?")
print("   → Data type: Integer")
print("   → Values: {0, 1, 2, 3, ...}")
print("   → Cannot be negative")

print("\n4️⃣  Success criteria?")
print("   → 'Predictions reasonably close to actual counts'")
print("   → Metric: **MAE or Poisson-based metrics**")

print("\n✓ FINAL ANSWER:")
print("   Problem Type: **COUNT REGRESSION**")
print("   Target Variable: Number of purchases (0, 1, 2, ...)")
print("   Success Metric: MAE")
print("   Algorithms: Poisson Regression, Negative Binomial, or Tree-based models")

# SCENARIO D: Content Platform Topic Tagging
print("\n\n" + "=" * 80)
print("SCENARIO D: Content platform wants article topic tagging")
print("=" * 80)

print("\nBusiness Requirement:")
print("  'Which topics apply to each article (Politics, Tech, Sports, etc)?'")
print("  'One article can cover MULTIPLE topics simultaneously'")

print("\n1️⃣  What am I predicting?")
print("   → Multiple categories that can apply simultaneously")
print("   → **CLASSIFICATION** (specifically: MULTI-LABEL)")

print("\n2️⃣  How many outcomes?")
print("   → Multiple labels NOT mutually exclusive")
print("   → Example: {Politics, Technology} OR {Sports, Entertainment}")

print("\n3️⃣  Target variable?")
print("   → Data type: Set or binary columns per label")
topics_ex = pd.DataFrame({
    'article_id': [1, 2, 3],
    'Politics': [1, 0, 1],
    'Technology': [1, 1, 0],
    'Sports': [0, 0, 1],
    'Entertainment': [0, 1, 0]
})
print("   → Example (binary encoding per label):")
print(topics_ex.to_string(index=False))

print("\n4️⃣  Success criteria?")
print("   → 'Accurately predict all applicable topics'")
print("   → Metric: **Hamming Loss, Subset Accuracy, F1 per label**")

print("\n✓ FINAL ANSWER:")
print("   Problem Type: **MULTI-LABEL CLASSIFICATION**")
print("   Target Variable: Set of applicable topics")
print("   Success Metric: Low Hamming Loss, High F1 per topic")
print("   Algorithms: One binary classifier per label, or Neural Network with sigmoid")

---

# Common Pitfalls in Problem Type Identification

## ❌ Pitfall 1: Treating Regression as Classification Unnecessarily

### The Mistake
You have continuous target (house prices) but bin into categories (Low, Medium, High) and treat as classification.

### Why It's Wrong
**You lose information.** A $310k house and $305k house are both "High" but are much closer to each other than a $500k house. Binning discards magnitude.

### When Binning IS Appropriate
- Business only needs broad categories (not precise values)
- Example: Recommendation system only cares "High Value," "Medium Value," "Low Value" for prioritization

---

## ❌ Pitfall 2: Treating Classification as Regression

### The Mistake
Encode categories as numbers (Setosa=1, Versicolor=2, Virginica=3) and use regression.

### Why It's Wrong
The numerical encoding implies ordering and magnitude that DON'T EXIST.
- Versicolor (2) is NOT "twice as much" as Setosa (1)
- Model learns nonsensical relationships

### Exception: Ordinal Classification
When categories have natural order (Very Dissatisfied → Dissatisfied → Neutral → Satisfied → Very Satisfied), regression can sometimes work (though ordinal regression is more principled).

---

## ❌ Pitfall 3: Ignoring Class Imbalance

### The Mistake
You have 99% negative, 1% positive class. Model achieves 99% accuracy. You celebrate.

### Why It's Wrong
**A model that ALWAYS predicts negative achieves 99% accuracy WITHOUT learning anything.**

Accuracy is **misleading metric when classes are imbalanced.**

### Solution
- Use **Precision, Recall, F1, ROC-AUC** instead of accuracy
- Consider class weighting or resampling techniques
- Monitor error rate on minority class specifically

---

## ❌ Pitfall 4: Confusing Multi-Class and Multi-Label

### The Mistake
Treat multi-label problem (movie can be Action AND Comedy) as multi-class (Action OR Comedy OR Drama, pick one).

### Why It's Wrong
You force model to choose ONE label when MULTIPLE may apply, discarding information.

### Solution
Use multi-label techniques: one binary classifier per label, or multi-label loss functions.

---

# Evaluation Metrics by Problem Type

## Classification Metrics

| Metric | Definition | Use Case |
|--------|-----------|----------|
| **Accuracy** | Fraction of correct predictions | When classes are balanced |
| **Precision** | Of predicted positives, fraction actually positive | When false positives are costly |
| **Recall (Sensitivity)** | Of actual positives, fraction correctly identified | When false negatives are costly |
| **F1 Score** | Harmonic mean of precision and recall | Balance both concerns |
| **ROC-AUC** | Area under ROC curve | Threshold-independent discrimination |
| **Confusion Matrix** | True positives, false positives, etc. | Understand error types |

### Example: Medical Diagnosis (Binary)
```
Disease Present: Positive class (1)
Disease Absent: Negative class (0)

High Recall (catch all diseases) → Few false negatives
High Precision (avoid false alarms) → Few false positives

Trade-off: Doctors often prefer HIGH RECALL (better to have false alarm than miss disease)
```

---

## Regression Metrics

| Metric | Definition | Use Case |
|--------|-----------|----------|
| **MAE** | Mean Absolute Error | Interpretable, robust to outliers |
| **MSE** | Mean Squared Error | Penalizes large errors more heavily |
| **RMSE** | Root Mean Squared Error | Same units as target variable |
| **R²** | Coefficient of Determination | Fraction of variance explained (0-1 scale) |
| **MAPE** | Mean Absolute Percentage Error | Percentage error (watch for small values) |

### Example: House Price Prediction
```
Prediction: $285,000
Actual: $290,000
Error: $5,000

MAE = $5,000 (easy to interpret)
MAPE = 1.7% (easy to interpret)
R² = 0.85 (model explains 85% of price variance)
```

---

# StockAxis: Applying Problem Type Framework to Your Project

## What We Know About StockAxis

From examining your codebase:

1. **Algorithm**: `RandomForestClassifier` ← This is a CLASSIFICATION model
2. **Evaluation Metrics**: Accuracy, Precision, Recall, F1, ROC-AUC ← Classification metrics
3. **Output Format**: Probability distributions via `predict_proba()` ← Classification output
4. **Target Column**: Configurable as `target_column` in `config.py`

## ✓ What We Can Confirm

**StockAxis is solving a SUPERVISED LEARNING CLASSIFICATION problem.**

The model:
- ✓ Learns from labeled historical data
- ✓ Predicts discrete categories (not continuous values)
- ✓ Outputs class probabilities
- ✓ Uses classification algorithms (Random Forest)
- ✓ Evaluates with classification metrics

## ? Questions to Answer About Your Specific Target

To fully define the problem, you need to examine your actual target variable:

### Question 1: Binary or Multi-Class?

**Check your data:**
```python
import pandas as pd

# Load your data
df = pd.read_csv('data/raw/dataset.csv')

# Analyze target
print(f"Unique values: {df['target'].unique()}")
print(f"Number of classes: {len(df['target'].unique())}")
print(f"Distribution:\n{df['target'].value_counts()}")
```

**Expected outcomes:**
- **Binary** (2 classes): Stock will go UP or DOWN tomorrow
- **Multi-class** (3+ classes): Market regime (BULL, NEUTRAL, BEAR)

### Question 2: Class Balance?

```python
print(df['target'].value_counts(normalize=True))
```

**If imbalanced (e.g., 80/20 split):**
→ Add `class_weight='balanced'` to RandomForestClassifier

### Question 3: What's Your Success Metric?

- Do you care equally about both classes? → Use Accuracy
- Must catch all buy signals? → Use Recall
- Must avoid false signals? → Use Precision
- Need overall ranking? → Use ROC-AUC

## Recommended Next Steps for StockAxis

1. [ ] Examine target variable in your dataset
2. [ ] Determine if binary or multi-class
3. [ ] Analyze class distribution
4. [ ] Define success metric based on trading goals
5. [ ] Update config if class imbalance detected
6. [ ] Train model with appropriate evaluation focus

---

# Key Takeaways: The Foundation is Everything

## Before you write any code, ask yourself:

1. **Am I predicting a category or a continuous value?**
   - Category → Classification
   - Number → Regression

2. **How many possible outcomes are there?**
   - 2 classes → Binary
   - 3+ classes → Multi-class
   - Multiple simultaneous → Multi-label

3. **Are outcomes mutually exclusive?**
   - Yes → Class standard classification/regression
   - No → Multi-label classification

4. **What does the target variable look like?**
   - Small set of unique values → Classification
   - Hundreds/thousands of unique values → Regression

5. **What does success look like for this problem?**
   - Business metric determines technical metric

## What Gets Wrong is Catastrophic

| Wrong Choice | Consequence |
|---|---|
| Regression model on classification → Nonsensical results |
| Classification on regression → Lost information |
| Wrong metrics on imbalanced data → Misleading conclusions |
| Multi-class on multi-label → Incomplete predictions |

## What Gets Right is Powerful

| Right Foundation | Enables |
|---|---|
| Clear problem type → Appropriate algorithms |
| Correct metrics → Honest evaluation |
| Deep understanding → Effective improvements |
| Sound definition → Deployment confidence |

---

## 🎯 Final Reflection

> **Problem type identification is not optional preparation. It is the foundation that determines whether everything that follows will be meaningful.**

Before you train any model, before you engineer any features, before you write any code:

**Structure your thinking. Define the problem. Then build the solution.**

Your StockAxis project already has the right foundation (classification framework). Now make sure your target variable, success metrics, and evaluation approach all align with your actual business goal.

---

## Quick Reference: Problem Type Checklist

### Before Training Any Model

- [ ] Target variable identified and examined
- [ ] Data type confirmed (discrete vs continuous)
- [ ] Classification or Regression decided
- [ ] If classification: Binary, Multi-class, or Multi-label?
- [ ] If regression: Standard, Count, or Bounded?
- [ ] Class distribution analyzed
- [ ] Success metric defined
- [ ] Algorithm families selected
- [ ] Appropriate evaluation metrics chosen

### Implementation Order (Always)

1. **Identify Problem Type** ← YOU ARE HERE
2. Define Success Metrics
3. Explore Target Distribution
4. Select Algorithms
5. Build Features
6. Train Model
7. Evaluate with Appropriate Metrics
8. Deploy with Confidence